In [94]:
import os
import chromadb
from openai import OpenAI
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [95]:
movies = [
    {
        "id": "1",
        "title": "Interstellar",
        "description": "Astronauts travel through space to save humanity.",
        "genre": "Sci-Fi",
        "year": 2014,
        "rating": "PG-13"
    },
    {
        "id": "2",
        "title": "Finding Nemo",
        "description": "A clownfish searches for his lost son.",
        "genre": "Animation",
        "year": 2003,
        "rating": "G"
    },
    {
        "id": "3",
        "title": "The Dark Knight",
        "description": "Batman fights crime in Gotham.",
        "genre": "Action",
        "year": 2008,
        "rating": "PG-13"
    },
    {
        "id": "4",
        "title": "Frozen",
        "description": "Two sisters discover the power of love.",
        "genre": "Animation",
        "year": 2013,
        "rating": "PG"
    },
    {
        "id": "5",
        "title": "The Matrix",
        "description": "A hacker discovers reality is a simulation.",
        "genre": "Sci-Fi",
        "year": 1999,
        "rating": "R"
    }
]

In [96]:
#getting the api key
load_dotenv()
Api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI(api_key=Api_key)
#chroma client
client = chromadb.PersistentClient(path="./chromdb")

In [97]:
# OpenAI embedding function
embedding_function = OpenAIEmbeddingFunction(
    api_key=Api_key ,
    model_name="text-embedding-3-small"
)

In [98]:
#deleting old collection to add a new one since metadata does not appear
try:
    client.delete_collection("movies")
    print("Old collection deleted.")
except:
    print("Collection doesn't exist yet.")



Collection doesn't exist yet.


In [99]:
#create collection
collection = client.get_or_create_collection(
    name ="movies",
    embedding_function=embedding_function
)

In [100]:
#loading data
metadatas = []
documents = []
ids = []

for movie in movies:
    document = f"""
     Title: {movie['title']}
     description: {movie['description']}
    
    """
    documents.append(document)

    ids.append(movie['id'])
    
    metadatas.append({
    "title": movie["title"],
    "genre": movie["genre"],
    "year": movie["year"],
    "rating": movie["rating"]
    })



     

In [101]:
#insert data
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
    
    
)


In [102]:
print(f"inserted {len(ids)} records")

inserted 5 records


In [ ]:
result = collection.query(
    query_texts=["space adventure"],
    where={}
    n_results=3
)

print(result)

{'ids': [['1', '5', '2']], 'embeddings': None, 'documents': [['\n     Title: Interstellar\n     description: Astronauts travel through space to save humanity.\n\n    ', '\n     Title: The Matrix\n     description: A hacker discovers reality is a simulation.\n\n    ', '\n     Title: Finding Nemo\n     description: A clownfish searches for his lost son.\n\n    ']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'year': 2014, 'title': 'Interstellar', 'genre': 'Sci-Fi', 'rating': 'PG-13'}, {'year': 1999, 'rating': 'R', 'genre': 'Sci-Fi', 'title': 'The Matrix'}, {'genre': 'Animation', 'title': 'Finding Nemo', 'year': 2003, 'rating': 'G'}]], 'distances': [[0.6025944948196411, 0.7000992894172668, 0.7530050873756409]]}


In [104]:
for i in range(len(result["ids"][0])):
    print("=" * 50)
    print("ID:", result["ids"][0][i])
    print("Metadata:", result["metadatas"][0][i])
    print("Document:", result["documents"][0][i])
    print("Distance:", result["distances"][0][i])

ID: 1
Metadata: {'year': 2014, 'title': 'Interstellar', 'genre': 'Sci-Fi', 'rating': 'PG-13'}
Document: 
     Title: Interstellar
     description: Astronauts travel through space to save humanity.

    
Distance: 0.6025944948196411
ID: 5
Metadata: {'year': 1999, 'rating': 'R', 'genre': 'Sci-Fi', 'title': 'The Matrix'}
Document: 
     Title: The Matrix
     description: A hacker discovers reality is a simulation.

    
Distance: 0.7000992894172668
ID: 2
Metadata: {'genre': 'Animation', 'title': 'Finding Nemo', 'year': 2003, 'rating': 'G'}
Document: 
     Title: Finding Nemo
     description: A clownfish searches for his lost son.

    
Distance: 0.7530050873756409
